# How long each program takes to run

This tool removes manual line breaks from Markdown prose. It has two
implementations, one in Python and one in Rust, and for every input the shared
conformance corpus covers they produce identical bytes. The only difference
between them is how long they take to run. This notebook measures that
difference.

There is no single number for it. Three costs make up the total, and they
differ from each other. Starting the program costs a fixed amount, once per
run. Handling one more file adds a small amount for each file. Rewriting the
text inside a file costs more as the file holds more text.

Which of the three accounts for most of the total depends on how the tool is
invoked. For a pre-commit hook on one edited file, starting the program is
almost all of it. For a run over every Markdown file in a repository, the
per-file cost is. For a run over one long document, rewriting the text is.

Three later sections time a third program on the same documents. Prettier's
`--prose-wrap never` setting joins soft-wrapped paragraphs as one part of a
general Markdown formatter, so a repository that already runs Prettier has the
transform without this tool. Those sections time it through its command line
and through `pre-commit`, and they keep the documents where its answer matches
this tool's apart from the documents where it does not.

Every number and every chart below is computed by the code cell above it, so
nothing in this text can contradict the output next to it. A number written
into a sentence would be wrong the next time the notebook runs.

In [1]:
import platform
import re
import subprocess
import sys
from pathlib import Path


def git(*argv: str) -> str:
    """Return the stripped stdout of a git command, or ``''`` if it failed."""
    return subprocess.run(
        ['git', *argv], capture_output=True, text=True, check=False
    ).stdout.strip()


REPO = Path(git('rev-parse', '--show-toplevel'))
PYTHON_CLI = REPO / '.venv/bin/unwrap-markdown-prose-py'
RUST_CLI = REPO / 'target/release/unwrap-markdown-prose-rs'

for path in (PYTHON_CLI, RUST_CLI):
    if not path.exists():
        raise SystemExit(f'missing {path}: run `uv sync` and `cargo build --release`')

# The Python that matters is the one the measured tool runs under, not the one
# executing this notebook, and the two are not always the same: this notebook
# has been executed by an unrelated 3.12 kernel while every timing below still
# came from the interpreter in this repository's virtual environment. Reporting
# the kernel would have credited these timings to an interpreter that never ran
# them. The console script's shebang names the interpreter that actually does.
shebang = PYTHON_CLI.read_text(encoding='utf-8').splitlines()[0]
TOOL_PYTHON = subprocess.run(
    [
        shebang.lstrip('#!').strip(),
        '-c',
        'import platform; print(platform.python_version())',
    ],
    capture_output=True,
    text=True,
    check=False,
).stdout.strip()

# The optimization level is read from Cargo.toml, not copied into this notebook.
# The release profile is set to optimize for binary size, so every Rust timing
# below is the timing of a size-optimized build. Anyone comparing against their
# own build needs to know which setting produced these numbers.
level = re.search(
    r'^\s*opt-level\s*=\s*(\S+)',
    (REPO / 'Cargo.toml').read_text(encoding='utf-8'),
    re.MULTILINE,
)
OPT_LEVEL = level.group(1).strip('"') if level else 'default'

rustc = subprocess.run(
    ['rustc', '--version'], capture_output=True, text=True, check=False
).stdout.strip()

print(f'machine   {platform.machine()}  {platform.system()} {platform.release()}')
print(f'python    {TOOL_PYTHON} runs the measured tool')
if platform.python_version() != TOOL_PYTHON:
    print(
        f'          {platform.python_version()} is executing this notebook, '
        'which affects the charts but none of the timings'
    )
print(f'rust      {rustc}')
# No revision is printed. Any hash this run could name would be the parent of
# the commit carrying the output, since that commit cannot exist while the run
# is happening, and it would be an older commit every time the page was read
# afterwards. These numbers come from the two programs rather than from the
# tree around them, and `git log` on this file says when they were measured.
BINARY_KIB = RUST_CLI.stat().st_size / 1024
tuning = ', optimized for size rather than speed' if OPT_LEVEL in {'z', 's'} else ''

print(f'binary    {BINARY_KIB:.0f} KiB at opt-level {OPT_LEVEL!r}{tuning}')

machine   arm64  Darwin 23.5.0
python    3.13.7 runs the measured tool
rust      rustc 1.86.0 (05f9846f8 2025-03-31)
binary    361 KiB at opt-level "'z'"


## Method

Each measurement times one complete run of the program, because that is what a
pre-commit hook or a CI step actually costs. Three warm-up runs are discarded,
and the runs after them are timed.

**Both the minimum and the median are reported.** The minimum is the better
estimate of what the program costs, because it is the run least affected by
other work happening on the machine at the same time. The median is printed
next to it so that the difference between the two is visible. A large
difference means the machine was busy, and a cell below checks for this and
reports it.

**The cost of starting a process is measured, not subtracted.** Every number
here includes the time to start a process: the operating system call that
creates it, the one that loads the program, and the pipe setup that this
notebook's own timing loop adds. That time is very small compared to the Python
total, and large compared to the Rust total. Dividing one by the other
therefore understates how much faster Rust is.

Subtracting that time would be less accurate than reporting it. The next cell
times three programs that do almost nothing, and they differ from each other by
more than the entire Rust runtime, so there is no single correct value to
subtract. Instead the cheapest of them is reported as a floor, drawn on the
charts, and used to mark any ratio whose smaller number is close to that floor.
Those ratios are labeled as lower bounds: the real difference is larger than
the number shown.

**The two implementations are compared inside the benchmark.** For every run,
the timing loop records a hash of the output and the exit code, then compares
both between Python and Rust. Without this check, a program that rejected its
arguments and exited immediately would record a very fast time and look like an
improvement. Recording a hash for every run, rather than keeping only the last
one, also detects a file being modified while the benchmark is reading it.

In [2]:
import hashlib
import statistics
import time
from dataclasses import dataclass

WARMUP, REPS = 3, 30

# How far above the minimum the median may be, as a percentage, before a
# measurement is reported as unreliable. Set above the run-to-run width of the
# startup rows, which are the most repeatable measurements here, rather than
# derived from them: the check below reports their spread as evidence instead
# of using it as the tolerance, because a tolerance drawn from three
# sub-millisecond rows would loosen every other row on the run where one of
# them happened to stutter.
SPREAD_LIMIT = 15.0

# How close to the floor the smaller of two times may be before their ratio is
# reported as a lower bound instead of a value.
FLOOR_FACTOR = 2.0

# Marks a ratio as a lower bound. Defined here as a name because an escape
# sequence cannot appear inside an f-string expression before Python 3.12.
BOUND = '\u2265'


@dataclass(frozen=True, slots=True)
class Timing:
    """Every sample from one measured command, and what those runs returned."""

    samples: tuple[float, ...]
    codes: frozenset[int]
    # One hash per repetition, rather than one saved copy of the output. If two
    # runs of the same command return different bytes, something changed the
    # files while the benchmark was reading them. Saving only one copy of the
    # output cannot show that, because the copy saved is the last run.
    digests: frozenset[str]

    @property
    def min(self) -> float:
        """Return the run least affected by other work on the machine."""
        return min(self.samples)

    @property
    def median(self) -> float:
        """Return the middle sample, which is raised by a busy machine."""
        return statistics.median(self.samples)

    @property
    def spread(self) -> float:
        """Return how far the median is above the minimum, as a percentage."""
        return (self.median - self.min) / self.min * 100

    @property
    def spawn_bound(self) -> bool:
        """Return whether this time is mostly the cost of starting a process."""
        return self.min < SPAWN_FLOOR * FLOOR_FACTOR


subprocess_run = subprocess.run
perf_counter = time.perf_counter
sha256 = hashlib.sha256


def measure(
    argv: list[str],
    reps: int = REPS,
    *,
    stdin: bytes | None = None,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
) -> Timing:
    """Time ``reps`` complete runs of ``argv``, after ``WARMUP`` discarded runs.

    Every run reads ``stdin`` as its standard input and starts in ``cwd`` with
    ``env`` as its environment.
    """
    for _ in range(WARMUP):
        subprocess_run(
            argv, input=stdin, capture_output=True, check=False, cwd=cwd, env=env
        )
    samples, codes, digests = [], set(), set()
    append_to_samples = samples.append
    add_to_codes = codes.add
    add_to_digests = digests.add
    for _ in range(reps):
        started = perf_counter()
        done = subprocess_run(
            argv, input=stdin, capture_output=True, check=False, cwd=cwd, env=env
        )
        append_to_samples((perf_counter() - started) * 1000)
        add_to_codes(done.returncode)
        add_to_digests(sha256(done.stdout).hexdigest())
    return Timing(tuple(samples), frozenset(codes), frozenset(digests))


# Three programs that do almost nothing, timed through the same loop as
# everything else. The cheapest is used as the floor. All three are printed
# because the differences between them are the reason none of them is
# subtracted from the measurements.
REFERENCES = ['/bin/echo', 'x'], ['/usr/bin/true'], ['/usr/bin/printf', '']

reference = {__: measure(argv) for argv in REFERENCES if Path(__ := argv[0]).exists()}
FLOOR = min(reference.values(), key=lambda timing: timing.min)
SPAWN_FLOOR = FLOOR.min

for name, timing in sorted(reference.items(), key=lambda item: item[1].min):
    print(f'{name:<18} {timing.min:>6.2f} ms   (median {timing.median:>5.2f} ms)')
spread = max(t.min for t in reference.values()) - SPAWN_FLOOR
print(f'\nfloor              {SPAWN_FLOOR:>6.2f} ms, the cheapest of the three')
print(f'they differ by     {spread:>6.2f} ms, which is why none of them is subtracted')

/usr/bin/true        1.13 ms   (median  1.18 ms)
/bin/echo            1.15 ms   (median  1.44 ms)
/usr/bin/printf      1.27 ms   (median  1.36 ms)

floor                1.13 ms, the cheapest of the three
they differ by       0.14 ms, which is why none of them is subtracted


In [3]:
scratch = Path('/tmp/markdown-prose-bench')
scratch.mkdir(exist_ok=True)

# One paragraph broken across several lines is the unit of work this tool
# exists to undo.
PARAGRAPH = 'A paragraph that has been hard\nwrapped across three\nseparate lines.\n\n'
(scratch / 'tiny.md').write_text(PARAGRAPH)

# Every file timed below is generated here rather than read from this
# repository, so what the timings describe is the two implementations rather
# than the tree around them, and adding a Markdown file to this repository does
# not make any figure on this page wrong. It also lets the number of files be
# varied, which the tree does not allow.
bulk = scratch / 'bulk'
bulk.mkdir(exist_ok=True)
for index in range(2000):
    (bulk / f'{index}.md').write_text(PARAGRAPH * 3)

# One large document. Startup is a negligible part of the time taken to process
# it, so what remains is the speed of the transform itself.
large = scratch / 'large.md'
large.write_text(PARAGRAPH * 60000)

print(f'generated file: {len(PARAGRAPH) * 3} bytes')
print(f'large document: {large.stat().st_size / 1e6:.1f} MB')

generated file: 207 bytes
large document: 4.1 MB


In [4]:
from typing import Literal

from IPython.display import Markdown, display


@dataclass(frozen=True, slots=True)
class Scenario:
    """One row of the table below: what was processed, and how long each took."""

    label: str
    python: Timing
    rust: Timing

    @property
    def timings(self) -> tuple[tuple[str, Timing], tuple[str, Timing]]:
        """Return each timing with the name of the program that produced it."""
        return ('python', self.python), ('rust', self.rust)


def file_list(count: int) -> tuple[Literal['--files-from'], str]:
    """Return arguments naming the first ``count`` of the generated files."""
    path = scratch / f'bulk-{count}.txt'
    path.write_text(
        '\n'.join(str(bulk / f'{index}.md') for index in range(count)) + '\n'
    )
    return '--files-from', str(path)


# One run over the 4 MB document takes about a second in Python, so that row
# uses fewer repetitions than the others. Stated here rather than left for a
# reader to work out from how long the cell takes.
LARGE_REPS = 10

scenarios = (
    ('one file', [str(scratch / 'tiny.md')], REPS),
    ('a typical commit: 5 files', file_list(5), REPS),
    ('a large repository: 2000 files', file_list(2000), REPS),
    (f'one {large.stat().st_size / 1e6:.0f} MB document', [str(large)], LARGE_REPS),
)

results: list[Scenario] = []
append_to_results = results.append
for label, args, reps in scenarios:
    append_to_results(
        Scenario(
            label=label,
            python=measure([str(PYTHON_CLI), *args, '--json'], reps),
            rust=measure([str(RUST_CLI), *args, '--json'], reps),
        )
    )


def ratio_cell(row: Scenario) -> str:
    """Return Python divided by Rust, marked as a bound when Rust is near the floor."""
    value = row.python.min / row.rust.min
    mark = BOUND if row.rust.spawn_bound else ''
    return f'{mark}{value:.1f}x'


table = [
    '| what is being processed | Python min / median | Rust min / median | Python is |',
    '| -- | --: | --: | --: |',
]
append_to_table = table.append
for row in results:
    append_to_table(
        f'| {row.label} '
        f'| {row.python.min:.1f} / {row.python.median:.1f} ms '
        f'| {row.rust.min:.1f} / {row.rust.median:.1f} ms '
        f'| {ratio_cell(row)} slower |'
    )
append_to_table('')
append_to_table(
    f'{BOUND} means the Rust time is within {FLOOR_FACTOR:.0f} times the '
    f'{SPAWN_FLOOR:.2f} ms floor, so most of what it measures is the cost of starting '
    'any process at all. Those ratios are lower bounds, and the real difference is '
    'larger.'
)
display(Markdown(chr(10).join(table)))

# Three separate questions, because they have three separate answers, and only
# the first of them can invalidate a recommendation.
#
#   AGREE    did the two implementations return the same bytes and the same
#            exit code? This is the assumption the whole document depends on.
#   STEADY   did each command return the same result on every repetition? It
#            may not: every file read here is generated into one scratch
#            directory, and a second run of this notebook writing there while
#            this one reads appears as a read error in one repetition out of
#            thirty.
#   CLEAN    did every run exit with status 0? A consistent non-zero status
#            means the tool reported a file it could not read, which is a fact
#            about the files rather than about either implementation.
AGREE = all(
    row.python.digests == row.rust.digests and row.python.codes == row.rust.codes
    for row in results
)
unsteady = [
    (row.label, name)
    for row in results
    for name, timing in row.timings
    if len(timing.digests) > 1 or len(timing.codes) > 1
]
CLEAN = all(
    timing.codes == frozenset({0}) for row in results for _, timing in row.timings
)
# A time that measures more than process startup is held to SPREAD_LIMIT. A time
# that is mostly startup is reported separately and held to nothing: the table
# already marks its ratio as a lower bound, and asking a reader to measure it
# again reproduces the same width, which was checked across three runs.
#
# The reference programs' own spread is reported as the evidence for that rather
# than used as the tolerance. It is three sub-millisecond measurements, and the
# widest of them has ranged from single digits to over 40% between runs of this
# notebook, so a tolerance drawn from it would loosen every row on the run where
# one reference program happened to stutter.
FLOOR_SPREAD = max(timing.spread for timing in reference.values())

over = [
    (row.label, name, timing)
    for row in results
    for name, timing in row.timings
    if timing.spread > SPREAD_LIMIT
]
noisy = [entry for entry in over if not entry[2].spawn_bound]
startup_bound = [entry for entry in over if entry[2].spawn_bound]

if AGREE:
    print('Both implementations returned identical bytes and identical exit codes for')
    print('every row above. The only difference between them here is speed.')
else:
    print('The implementations returned different results. Ignore every')
    print('recommendation below.')
    for row in results:
        if row.python.digests != row.rust.digests:
            print(f'     different output: {row.label}')
        if row.python.codes != row.rust.codes:
            print(f'     different exit status: {row.label}')

if unsteady:
    print('\nA command did not return the same result on every repetition, so a file')
    print('changed while the benchmark was running. The timings are still usable, but')
    print('these rows are worth measuring again:')
    for label, impl in unsteady:
        print(f'     {impl:<7} {label}')
elif not CLEAN:
    print('\nEvery repetition of one row exited with a non-zero status, identically in')
    print('both implementations. The tool reported a file it could not read. That is a')
    print('fact about the files, not a difference between the implementations.')
    for row in results:
        codes = row.python.codes | row.rust.codes
        if codes != frozenset({0}):
            print(f'     {row.label}: exit {sorted(codes)}')

if noisy:
    print(f'\nIn these rows the median is more than {SPREAD_LIMIT:.0f}% above the')
    print('minimum, and the row measures more than process startup, so the')
    print('machine was busy. Measure them again:')
    for label, impl, timing in noisy:
        print(f'     {impl:<7} {label:<44} +{timing.spread:.0f}%')
else:
    print('\nEvery row that measures more than starting a process is within')
    print(f'{SPREAD_LIMIT:.0f}% of its own minimum.')

if startup_bound:
    print(f'\nThese are wider than {SPREAD_LIMIT:.0f}%, and are expected to be.')
    print(f'Their time is mostly the {SPAWN_FLOOR:.2f} ms of starting a process,')
    print(f'which varied by {FLOOR_SPREAD:.0f}% this run while doing nothing. The')
    print(f'table marks their ratios {BOUND} for the same reason:')
    for label, impl, timing in startup_bound:
        print(f'     {impl:<7} {label:<44} +{timing.spread:.0f}%')

| what is being processed | Python min / median | Rust min / median | Python is |
| -- | --: | --: | --: |
| one file | 27.7 / 28.5 ms | 1.9 / 2.3 ms | ≥14.3x slower |
| a typical commit: 5 files | 32.4 / 35.0 ms | 1.9 / 2.3 ms | ≥17.1x slower |
| a large repository: 2000 files | 176.8 / 184.9 ms | 36.8 / 37.6 ms | 4.8x slower |
| one 4 MB document | 885.1 / 913.8 ms | 86.0 / 90.6 ms | 10.3x slower |

≥ means the Rust time is within 2 times the 1.13 ms floor, so most of what it measures is the cost of starting any process at all. Those ratios are lower bounds, and the real difference is larger.

Both implementations returned identical bytes and identical exit codes for
every row above. The only difference between them here is speed.

Every row that measures more than starting a process is within
15% of its own minimum.

These are wider than 15%, and are expected to be.
Their time is mostly the 1.13 ms of starting a process,
which varied by 25% this run while doing nothing. The
table marks their ratios ≥ for the same reason:
     rust    one file                                     +20%
     rust    a typical commit: 5 files                    +24%


## What the time is spent on

The table above combines two separate costs: a fixed cost paid once per run,
and a cost paid for each file. Telling them apart is what makes it possible to
recommend one implementation per way of using the tool.

The next cell measures the fixed cost directly. It times the cheapest process
this machine can start, a Python interpreter that does nothing, the same
interpreter after importing this tool, and then each of the two programs on a
single file.

The difference between the two interpreter rows is the cost of importing the
tool. The difference between the last two rows is what one run of the Rust
program saves. The first row is the cost that neither implementation can go
below, because it is what starting any process costs.

In [5]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.figure import Figure

# Element ids are derived from this value rather than from memory addresses, so
# two runs over the same numbers produce the same file.
mpl.rcParams['svg.hashsalt'] = 'markdown-prose-hooks'


def hide_spines(*all_axes: Axes) -> None:
    """Remove the top and right borders from each chart."""
    for axes in all_axes:
        axes.spines['top'].set_visible(False)
        axes.spines['right'].set_visible(False)


def save_chart(figure: Figure, name: str) -> None:
    """Write ``figure`` to the docs directory as the bytes that get committed.

    Two things would otherwise change the file on every run, whatever was
    measured. matplotlib writes the current date into the SVG metadata, and it
    leaves trailing spaces on a few hundred lines, which this repository
    removes with its own hook when the file is committed. Left as they are, the
    committed chart is not the file this notebook wrote, running the notebook
    always leaves uncommitted changes, and a real change to a measurement is
    hard to see among the changes that mean nothing.
    """
    path = REPO / 'docs' / 'notebooks' / 'benchmarks' / name
    figure.savefig(path, metadata={'Date': None})
    plt.close(figure)
    text = path.read_text(encoding='utf-8')
    path.write_text(
        chr(10).join(line.rstrip() for line in text.split(chr(10))),
        encoding='utf-8',
    )
    print(f'chart written to docs/notebooks/benchmarks/{name}')


floor_rows = {
    'cheapest process': FLOOR,
    'Python interpreter, doing nothing': measure([sys.executable, '-c', 'pass']),
    'Python interpreter, after import': measure(
        [sys.executable, '-c', 'import markdown_prose_hooks.unwrap']
    ),
    'Python program, one file': measure(
        [str(PYTHON_CLI), str(scratch / 'tiny.md'), '--json']
    ),
    'Rust program, one file': measure(
        [str(RUST_CLI), str(scratch / 'tiny.md'), '--json']
    ),
}
for label, timing in floor_rows.items():
    print(f'{label:<36} {timing.min:>6.2f} ms   (median {timing.median:>6.2f} ms)')

python_startup = floor_rows['Python program, one file'].min
rust_startup = floor_rows['Rust program, one file'].min
saving = python_startup - rust_startup

print(f'\nPython takes {python_startup / rust_startup:.1f} times as long to start')
if rust_startup > SPAWN_FLOOR:
    corrected = (python_startup - SPAWN_FLOOR) / (rust_startup - SPAWN_FLOOR)
    print(f'With the floor removed from both, it would be {corrected:.1f} times.')
    print(
        f'   That figure is shown but not used. The Rust time is only '
        f'{rust_startup / SPAWN_FLOOR:.1f} times'
    )
    print('   the floor, so removing the floor divides by the small difference between')
    print('   two similar numbers, and the result changes greatly when the floor')
    print('   changes slightly. Treat the figure above as a minimum instead.')
else:
    print('   The Rust program was as fast as the cheapest process this machine can')
    print('   start, so its own startup cost is below what this method can measure.')
    print('   The figure above is a minimum.')

print(f'\nEach run of the Rust program saves {saving:.1f} ms.')
print(f'Over a hundred runs a day that is {saving * 100 / 1000:.1f} seconds.')
print('   Subtracting is reliable here even though dividing is not. The cost of')
print('   starting a process is the same in both numbers, so it cancels out when')
print('   they are subtracted.')

figure, axes = plt.subplots(figsize=(8, 3.2))
labels = list(floor_rows)
axes.barh(
    labels,
    [floor_rows[label].min for label in labels],
    color=['#999999', '#bbbbbb', '#bbbbbb', '#1f77b4', '#ff7f0e'],
)
axes.axvline(SPAWN_FLOOR, color='#444444', linestyle='--', linewidth=1)
axes.set_xscale('log')
axes.set_xlabel('time (milliseconds, logarithmic scale)')
axes.set_title('The dashed line is what starting any process costs')
axes.invert_yaxis()
for index, label in enumerate(labels):
    axes.text(
        floor_rows[label].min * 1.15,
        index,
        f'{floor_rows[label].min:.2f} ms',
        va='center',
        fontsize=9,
    )
axes.set_xlim(right=max(t.min for t in floor_rows.values()) * 3)
axes.grid(alpha=0.3, axis='x')
hide_spines(axes)
figure.tight_layout()
print()
save_chart(figure, 'benchmarks-startup.svg')

cheapest process                       1.13 ms   (median   1.18 ms)
Python interpreter, doing nothing     12.96 ms   (median  13.32 ms)
Python interpreter, after import      26.18 ms   (median  27.01 ms)
Python program, one file              27.84 ms   (median  29.03 ms)
Rust program, one file                 1.79 ms   (median   2.17 ms)

Python takes 15.5 times as long to start
With the floor removed from both, it would be 40.1 times.
   That figure is shown but not used. The Rust time is only 1.6 times
   the floor, so removing the floor divides by the small difference between
   two similar numbers, and the result changes greatly when the floor
   changes slightly. Treat the figure above as a minimum instead.

Each run of the Rust program saves 26.0 ms.
Over a hundred runs a day that is 2.6 seconds.
   Subtracting is reliable here even though dividing is not. The cost of
   starting a process is the same in both numbers, so it cancels out when
   they are subtracted.

chart written 

![The dashed line is what starting any process costs](benchmarks-startup.svg)

## Separating the fixed cost from the cost per file

Running the tool once over an increasing number of files separates the two
costs. In the left chart, where a line begins is the fixed cost, and how
steeply it rises is the cost per file. The right chart shows how many times
slower Python is at each number of files.

Both axes on the left are logarithmic because the number of files spans three
orders of magnitude. On ordinary axes, six of the eight measurements would fall
within the first quarter of the chart, where the fixed cost cannot be read.

In [6]:
# Fewer repetitions than the table above. This cell and the next one exist to
# show the shape of a curve rather than to produce a figure quoted elsewhere,
# and the shape is already stable at this number of repetitions.
SWEEP_REPS = 15

counts = 1, 10, 50, 100, 250, 500, 1000, 2000
curve = {'Python': [], 'Rust': []}
python_curve, rust_curve = curve['Python'], curve['Rust']
append_to_python_curve = python_curve.append
append_to_rust_curve = rust_curve.append
for count in counts:
    args = file_list(count)
    append_to_python_curve(measure([str(PYTHON_CLI), *args, '--json'], SWEEP_REPS).min)
    append_to_rust_curve(measure([str(RUST_CLI), *args, '--json'], SWEEP_REPS).min)

figure, (left, right) = plt.subplots(1, 2, figsize=(11, 4.2))
plot_left = left.plot
for name, series in curve.items():
    plot_left(counts, series, marker='o', label=name)
left.axhline(SPAWN_FLOOR, color='#444444', linestyle='--', linewidth=1)
left.text(counts[0], SPAWN_FLOOR * 1.15, 'floor', fontsize=8, color='#444444')
left.set_xscale('log')
left.set_yscale('log')
left.set_xlabel('files processed in one run')
left.set_ylabel('time (milliseconds)')
left.set_title('Where a line starts is the fixed cost')
left.legend()
left.grid(alpha=0.3, which='both')

ratios = [p / r for p, r in zip(python_curve, rust_curve, strict=True)]
right.plot(counts, ratios, marker='o', color='#2ca02c')
right.set_xscale('log')
right.set_ylim(bottom=0)
right.set_xlabel('files processed in one run')
right.set_ylabel('times slower')
right.set_title('Python, relative to Rust')
right.grid(alpha=0.3, which='both')
hide_spines(left, right)

# The chart is written next to the notebook rather than embedded in it, and as
# SVG rather than PNG. An embedded PNG is stored as base64 text, which the two
# spell-checking hooks read as prose: measured on thirty different versions of
# this chart, codespell reported a misspelling in eighteen of them, because
# base64 splits into thousands of short letter sequences and some of them match
# dictionary entries. The same thirty charts written as SVG produced no
# findings from either hook. A PNG file would be worse still, because this
# repository routes `*.png` through Git LFS, so the chart would be committed as
# a pointer and appear broken to anyone cloning without git-lfs installed.
figure.tight_layout()
save_chart(figure, 'benchmarks.svg')

# A straight line between the first and last points is only a fair summary if
# the measurements between them are also close to that line. Every consecutive
# pair is printed so that a reader can see whether they are, rather than being
# asked to assume it.
count_marginal, count_intercept = {}, {}
print(f'{"":<8} {"fixed cost":>11} {"per file":>12}   cost per file between each pair')
for name, series in curve.items():
    pairs = [
        (series[i + 1] - series[i]) / (counts[i + 1] - counts[i]) * 1000
        for i in range(len(counts) - 1)
    ]
    count_intercept[name] = series[0]
    count_marginal[name] = (series[-1] - series[0]) / (counts[-1] - counts[0]) * 1000
    print(
        f'{name:<8} {series[0]:>8.1f} ms {count_marginal[name]:>8.1f} us   '
        + ' '.join(f'{value:.0f}' for value in pairs)
    )
print('\nThe leftmost pairs are the least reliable, because they are differences')
print('between measurements only a few milliseconds apart. They stabilize once the')
print('per-file work is large enough to measure, and that is what makes the single')
print('figure in the third column a fair summary rather than a straight line drawn')
print('through a curve.')
print(
    f'\nPython is {ratios[0]:.1f} times slower at {counts[0]} file and '
    f'{ratios[-1]:.1f} times slower at {counts[-1]} files.'
)

chart written to docs/notebooks/benchmarks/benchmarks.svg
          fixed cost     per file   cost per file between each pair
Python       28.4 ms     76.6 us   301 104 5 74 77 87 72
Rust          1.6 ms     18.8 us   101 -1 17 17 24 14 20

The leftmost pairs are the least reliable, because they are differences
between measurements only a few milliseconds apart. They stabilize once the
per-file work is large enough to measure, and that is what makes the single
figure in the third column a fair summary rather than a straight line drawn
through a curve.

Python is 17.7 times slower at 1 file and 4.6 times slower at 2000 files.


![Where a line starts is the fixed cost](benchmarks.svg)

## What the cost per file depends on

The cost of one more file is not a fixed number, and it is not a property of
the tool. Opening a file is a request to the operating system, and costs about
the same in both languages. Reading its contents and rewriting the paragraphs
is the work the two implementations do differently. So how much slower Python
is per file depends on how much text an average file contains.

The next cell measures that. For each file size it times two runs, one over a
small number of files and one over a large number, and subtracts the first from
the second. Subtracting removes the fixed startup cost exactly, without having
to estimate it. That matters most for small files, where startup would
otherwise be most of what was measured.

The result is a single curve. At one end, processing a file costs little more
than opening it. At the other end, the cost is almost entirely the work of
rewriting the text. Any figure quoted for the difference per file is one point
on this curve, and which point applies to a repository depends on the
Markdown files in it.

In [7]:
# Two numbers of files for each file size, and the difference between them. The
# fixed startup cost is identical in both, so subtracting removes it exactly
# and nothing here needs an estimate of it.
SIZE_REPS = 8
LOW, HIGH = 100, 500
paragraph_counts = 1, 3, 10, 30, 100


def listing(
    folder: Path, count: int
) -> tuple[Literal['--files-from'], str, Literal['--json']]:
    """Return arguments naming ``count`` files from ``folder``, asking for JSON."""
    path = scratch / f'{folder.name}-{count}.txt'
    path.write_text(
        '\n'.join(str(folder / f'{index}.md') for index in range(count)) + '\n'
    )
    return '--files-from', str(path), '--json'


def marginal_us(cli: Path, folder: Path) -> float:
    """Return the microseconds one more file from ``folder`` costs ``cli``.

    The fixed startup cost is identical in both measurements, so subtracting
    one from the other removes it exactly rather than by estimate.
    """
    low = measure([str(cli), *listing(folder, LOW)], SIZE_REPS).min
    high = measure([str(cli), *listing(folder, HIGH)], SIZE_REPS).min
    return (high - low) / (HIGH - LOW) * 1000


sweep = {'bytes': [], 'Python': [], 'Rust': []}
sweep_bytes = sweep['bytes']
sweep_python = sweep['Python']
sweep_rust = sweep['Rust']
append_to_sweep_bytes = sweep_bytes.append
append_to_sweep_python = sweep_python.append
append_to_sweep_rust = sweep_rust.append
for paragraphs in paragraph_counts:
    folder = scratch / f'size-{paragraphs}'
    folder.mkdir(exist_ok=True)
    for index in range(HIGH):
        (folder / f'{index}.md').write_text(PARAGRAPH * paragraphs)
    append_to_sweep_bytes(len(PARAGRAPH) * paragraphs)
    append_to_sweep_python(marginal_us(PYTHON_CLI, folder))
    append_to_sweep_rust(marginal_us(RUST_CLI, folder))

size_ratios = [p / r for p, r in zip(sweep_python, sweep_rust, strict=True)]

# No point on the curve is marked as the one that applies. Which point applies
# depends on the files a particular run is given, and marking this repository's
# own average would put a figure on the chart that moves whenever a Markdown
# file is added to the tree, while the curve itself would not have moved.
figure, axes = plt.subplots(figsize=(8, 4.5))
axes.plot(sweep_bytes, size_ratios, marker='o', color='#2ca02c')
axes.set_xscale('log')
axes.set_ylim(bottom=0)
axes.set_xlabel(
    f'bytes per file (logarithmic scale; each point subtracts {LOW} files from {HIGH})'
)
axes.set_ylabel('times slower, per file')
axes.set_title('How much slower Python is per file depends on the text in it')
axes.grid(alpha=0.3, which='both')
hide_spines(axes)
figure.tight_layout()
save_chart(figure, 'benchmarks-bytes.svg')

table = [
    '| bytes per file | Python | Rust | Python is |',
    '| --: | --: | --: | --: |',
]
append_to_table = table.append
for size, python_us, rust_us, value in zip(
    sweep_bytes, sweep_python, sweep_rust, size_ratios, strict=True
):
    append_to_table(
        f'| {size} | {python_us:.1f} us per file | {rust_us:.1f} us per file '
        f'| {value:.1f}x slower |'
    )
append_to_table('')
append_to_table(
    f'Across the {sweep["bytes"][-1] // sweep["bytes"][0]}-fold range of file sizes '
    f'measured, Python is between {min(size_ratios):.1f} and {max(size_ratios):.1f} '
    f'times slower per file. The figure that applies to a run is the one for the '
    f'size of the files in it, read off the row above or the curve beside it.'
)
display(Markdown(chr(10).join(table)))

# The previous cell measured this same quantity at one file size, because the
# files it used are 207 bytes each. Two independent measurements of one
# quantity are worth comparing rather than assuming they agree.
common = sweep['bytes'].index(len(PARAGRAPH) * 3)
print(
    f'Checked against the previous cell at {sweep["bytes"][common]} bytes per file, '
    'where both measured the same thing:'
)
for name in ('Python', 'Rust'):
    here, there = sweep[name][common], count_marginal[name]
    print(
        f'  {name:<7} {here:>6.1f} us here, {there:>6.1f} us there, a difference of '
        f'{abs(here - there) / there * 100:.0f}%'
    )

chart written to docs/notebooks/benchmarks/benchmarks-bytes.svg


| bytes per file | Python | Rust | Python is |
| --: | --: | --: | --: |
| 69 | 46.9 us per file | 13.7 us per file | 3.4x slower |
| 207 | 76.8 us per file | 18.6 us per file | 4.1x slower |
| 690 | 193.4 us per file | 27.5 us per file | 7.0x slower |
| 2070 | 448.2 us per file | 63.9 us per file | 7.0x slower |
| 6900 | 1453.3 us per file | 150.7 us per file | 9.6x slower |

Across the 100-fold range of file sizes measured, Python is between 3.4 and 9.6 times slower per file. The figure that applies to a run is the one for the size of the files in it, read off the row above or the curve beside it.

Checked against the previous cell at 207 bytes per file, where both measured the same thing:
  Python    76.8 us here,   76.6 us there, a difference of 0%
  Rust      18.6 us here,   18.8 us there, a difference of 1%


![How much slower Python is per file depends on the text in it](benchmarks-bytes.svg)

## What each implementation costs to install

Everything above measures how long the tool takes to run. Choosing between the
two also depends on what each costs to set up, because `pre-commit` builds an
environment for a hook the first time it runs it, and again whenever the pinned
revision changes.

The next cell measures both. For Python it creates a virtual environment and
installs the package into it, which is what `pre-commit` does for a
`language: python` hook. For Rust it compiles and installs the program, which
is what it does for a `language: rust` hook. Each is timed into an empty
directory, so nothing is reused between one run and the next.

One case is deliberately not measured here: a machine with no Rust toolchain.
There `pre-commit` downloads and installs a toolchain first, which costs far
more than anything on this page, and it is why the rule in the README turns on
whether cargo is already installed.

In [8]:
import os
import shutil
from collections.abc import Callable

# Three cold runs each. Fewer than the timings above because each run creates a
# whole environment, and because the numbers are seconds apart rather than
# milliseconds apart, so a small disturbance changes them proportionally less.
INSTALL_REPS = 3
installs = scratch / 'installs'


@dataclass(frozen=True, slots=True)
class InstallCost:
    """Seconds taken to create one environment, over several cold runs."""

    samples: tuple[float, ...]

    @property
    def min(self) -> float:
        """Return the fastest of the cold runs."""
        return min(self.samples)

    @property
    def median(self) -> float:
        """Return the middle cold run."""
        return statistics.median(self.samples)


subprocess_run = subprocess.run


def python_environment(target: Path) -> None:
    """Create a virtual environment in ``target`` and install this package."""
    subprocess_run(
        [sys.executable, '-m', 'venv', str(target)], check=True, capture_output=True
    )
    subprocess_run(
        [
            str(target / 'bin/pip'),
            'install',
            '--quiet',
            '--disable-pip-version-check',
            str(REPO),
        ],
        check=True,
        capture_output=True,
    )


def rust_environment(target: Path) -> None:
    """Compile the Rust program and install it into ``target``."""
    subprocess_run(
        [
            'cargo',
            'install',
            '--quiet',
            '--locked',
            '--root',
            str(target),
            '--path',
            str(REPO),
        ],
        check=True,
        capture_output=True,
        env={**os.environ, 'CARGO_TARGET_DIR': str(target / 'build')},
    )


rmtree = shutil.rmtree
perf_counter = time.perf_counter


def time_environment(setup: Callable[[Path], None]) -> InstallCost:
    """Time ``setup`` into an empty directory, ``INSTALL_REPS`` times."""
    samples = []
    append_to_samples = samples.append
    for index in range(INSTALL_REPS):
        target = installs / f'{setup.__name__}-{index}'
        target.parent.mkdir(parents=True, exist_ok=True)
        rmtree(target, ignore_errors=True)
        started = perf_counter()
        setup(target)
        append_to_samples(perf_counter() - started)
        rmtree(target, ignore_errors=True)
    return InstallCost(tuple(samples))


python_install = time_environment(python_environment)
rust_environment_cost = time_environment(rust_environment)

print(
    f'Python environment, venv plus pip install   {python_install.min:.2f} s   '
    f'(median {python_install.median:.2f} s)'
)
print(
    f'Rust environment, cargo install            '
    f'{rust_environment_cost.min:.2f} s   '
    f'(median {rust_environment_cost.median:.2f} s)'
)

# Below this fraction the two are treated as equal. Both figures are seconds
# spent creating files and compiling, which varies with what else the machine is
# doing, and repeated runs of this cell have put each one ahead of the other.
# Reporting a winner from a difference this small would claim a precision the
# measurement does not have.
SAME_WITHIN = 0.25

extra_install = rust_environment_cost.min - python_install.min
cheaper = min(python_install.min, rust_environment_cost.min)

print()
if abs(extra_install) / cheaper < SAME_WITHIN:
    print(
        f'The two differ by {abs(extra_install):.2f} s, which is within '
        f'{SAME_WITHIN:.0%} of the smaller'
    )
    print('of them. Setting up either environment costs about the same, and which')
    print('one comes out ahead changes between runs of this cell. Installing costs')
    print('nothing worth counting either way; running is where they differ.')
elif extra_install <= 0:
    print(f'Rust costs {-extra_install:.2f} s less to set up, and also runs faster.')
else:
    print(
        f'Rust costs {extra_install:.2f} s more to set up and saves '
        f'{saving:.1f} ms per run,'
    )
    runs = extra_install / (saving / 1000)
    print(f'so it costs less in total after {runs:.0f} runs.')
print()
print('   Both figures assume a warm package cache. A first install on a new')
print('   machine downloads more and takes longer, on either side.')

Python environment, venv plus pip install   2.08 s   (median 2.18 s)
Rust environment, cargo install            2.47 s   (median 2.57 s)

The two differ by 0.38 s, which is within 25% of the smaller
of them. Setting up either environment costs about the same, and which
one comes out ahead changes between runs of this cell. Installing costs
nothing worth counting either way; running is where they differ.

   Both figures assume a warm package cache. A first install on a new
   machine downloads more and takes longer, on either side.


## Prettier on the same documents

`prettier --prose-wrap never` joins soft-wrapped paragraphs too, and the parity
notebook, `docs/notebooks/prettier-parity/prettier-parity.ipynb`, measures
where its answers and this tool's differ. This section measures how long it
takes to reach them.

A time is only comparable where the answers are the same. A program that
returns a different document has done different work, and timing it against
this tool compares two jobs rather than two programs. So every document in this
section is one that all three programs return as the same bytes. The documents
hold the same paragraph as the ones above, but they end in one newline rather
than a blank line, because Prettier ends every file with exactly one newline
and that byte would be the only difference between the answers.

A single document is read on standard input and written to standard output,
which makes the three answers comparable byte for byte, and the cell checks
that they match on every repetition. The rows over many files cannot work that
way, so they pass the files by name and ask each program which of them it would
change, and the cell checks that every program names every file.

Prettier runs through its command line, as a person or a hook runs it, at the
version this repository's own Prettier hook pins. It is given no configuration
file, no editorconfig and an empty ignore file. The ignore file is the setting
that matters. Prettier returns a file its ignore file excludes unchanged,
exits 0 and prints nothing, so a run started in a directory whose ignore file names
`*.md` would record Prettier as the fastest program on this page. The byte
comparison is what would catch that.

In [9]:
import json

# The `rev:` on this repository's own Prettier hook, so the Prettier timed here
# is the one this repository runs rather than whatever the registry calls
# newest.
pinned = re.search(
    r'rbubley/mirrors-prettier\s*\n\s*rev:\s*v(\S+)',
    (REPO / '.pre-commit-config.yaml').read_text(encoding='utf-8'),
)
if pinned is None:
    raise SystemExit(f'{REPO / ".pre-commit-config.yaml"} pins no prettier')
PRETTIER_PIN = pinned.group(1)

node_prefix = scratch / 'node'
installed = subprocess_run(
    [
        'npm',
        'install',
        '--prefix',
        str(node_prefix),
        '--no-audit',
        '--no-fund',
        '--loglevel',
        'error',
        f'prettier@{PRETTIER_PIN}',
    ],
    capture_output=True,
    text=True,
    check=False,
)
if installed.returncode != 0:
    raise SystemExit(
        f'npm could not install prettier {PRETTIER_PIN}: {installed.stderr}'
    )

PRETTIER_SCRIPT = node_prefix / 'node_modules/prettier/bin/prettier.cjs'
PRETTIER_VERSION = subprocess_run(
    ['node', str(PRETTIER_SCRIPT), '--version'],
    capture_output=True,
    text=True,
    check=False,
).stdout.strip()
if PRETTIER_VERSION != PRETTIER_PIN:
    raise SystemExit(f'npm resolved prettier {PRETTIER_VERSION}, not {PRETTIER_PIN}')
NODE_VERSION = subprocess_run(
    ['node', '--version'], capture_output=True, text=True, check=False
).stdout.strip()

# Named rather than left to the working directory, and empty: Prettier returns
# a file its ignore file excludes unchanged, exits 0 and prints nothing. Every
# other setting is Prettier's default except the one this comparison is about.
empty_ignore = scratch / 'empty-ignore'
empty_ignore.write_text('')
PRETTIER = [
    'node',
    str(PRETTIER_SCRIPT),
    '--no-config',
    '--no-editorconfig',
    '--ignore-path',
    str(empty_ignore),
    '--prose-wrap',
    'never',
]
PROGRAMS = (
    ('Rust', [str(RUST_CLI)]),
    ('Python', [str(PYTHON_CLI)]),
    ('Prettier', PRETTIER),
)


def on_stdin(name: str, command: list[str], path: Path) -> list[str]:
    """Return arguments that make ``command`` rewrite ``path`` read on stdin."""
    if name == 'Prettier':
        return [*command, '--stdin-filepath', str(path)]
    return [*command, '-']


def by_name(name: str, command: list[str], paths: list[str]) -> list[str]:
    """Return arguments that make ``command`` report which ``paths`` would change."""
    if name == 'Prettier':
        return [*command, '--list-different', *paths]
    return [*command, '--json', *paths]


def names_every_file(name: str, command: list[str], paths: list[str]) -> bool:
    """Return whether ``command`` reports that every one of ``paths`` would change."""
    out = subprocess_run(
        by_name(name, command, paths), capture_output=True, text=True, check=False
    ).stdout
    if name == 'Prettier':
        return len(out.splitlines()) == len(paths)
    return sum(entry['changed'] for entry in json.loads(out)['files']) == len(paths)


def wrapped_document(paragraphs: int, head: str = '') -> str:
    """Return ``head`` and ``paragraphs`` wrapped paragraphs, ending in one newline."""
    return (head + PARAGRAPH * paragraphs).rstrip('\n') + '\n'


def multiple(slower: Timing, faster: Timing) -> str:
    """Return how many times as long ``slower`` took, marked as a bound if due."""
    mark = BOUND if faster.spawn_bound else ''
    return f'{mark}{slower.min / faster.min:.1f}x'


prettier_docs = scratch / 'prettier'
rmtree(prettier_docs, ignore_errors=True)
(prettier_docs / 'bulk').mkdir(parents=True)
one_doc = prettier_docs / 'one.md'
one_doc.write_text(wrapped_document(1))
# As many paragraphs as the large document above.
whole_doc = prettier_docs / 'large.md'
whole_doc.write_text(wrapped_document(60000))
bulk_docs = [str(prettier_docs / 'bulk' / f'{index}.md') for index in range(2000)]
for path in bulk_docs:
    Path(path).write_text(wrapped_document(3))

# Prettier takes seconds over each of the last two rows, so they are repeated
# as often as the large document is above.
prettier_scenarios = (
    ('one file', one_doc, [], REPS),
    ('a typical commit: 5 files', None, bulk_docs[:5], REPS),
    ('a large repository: 2000 files', None, bulk_docs, LARGE_REPS),
    (
        f'one {whole_doc.stat().st_size / 1e6:.0f} MB document',
        whole_doc,
        [],
        LARGE_REPS,
    ),
)

prettier_rows: list[tuple[str, dict[str, Timing]]] = []
unmatched: list[str] = []
for label, doc, paths, reps in prettier_scenarios:
    timings = {}
    for name, command in PROGRAMS:
        if doc is None:
            if not names_every_file(name, command, paths):
                unmatched.append(f'{name} did not name every file: {label}')
            timings[name] = measure(by_name(name, command, paths), reps)
        else:
            stdin = doc.read_bytes()
            timings[name] = measure(on_stdin(name, command, doc), reps, stdin=stdin)
            if timings[name].codes != frozenset({0}):
                unmatched.append(
                    f'{name} exited {sorted(timings[name].codes)}: {label}'
                )
    if doc is not None and len({timing.digests for timing in timings.values()}) != 1:
        unmatched.append(f'the three answers differ: {label}')
    prettier_rows.append((label, timings))

node_floor = measure(['node', '-e', ''])

table = [
    (
        '| what is being processed | Rust | Python | Prettier '
        '| Prettier / Rust | Prettier / Python |'
    ),
    '| -- | --: | --: | --: | --: | --: |',
]
append_to_table = table.append
for label, timings in prettier_rows:
    cells = ' | '.join(
        f'{timings[name].min:.1f} / {timings[name].median:.1f} ms'
        for name, _ in PROGRAMS
    )
    append_to_table(
        f'| {label} | {cells} '
        f'| {multiple(timings["Prettier"], timings["Rust"])} '
        f'| {multiple(timings["Prettier"], timings["Python"])} |'
    )
append_to_table('')
append_to_table(
    'Each time is the minimum and then the median. '
    f'{BOUND} means the Rust time is within {FLOOR_FACTOR:.0f} times the '
    f'{SPAWN_FLOOR:.2f} ms floor, so that ratio is a lower bound.'
)
display(Markdown(chr(10).join(table)))

print(f'prettier  {PRETTIER_VERSION}')
print(f'node      {NODE_VERSION}')
print(
    f'          {node_floor.min:.1f} ms to run an empty script '
    f'(median {node_floor.median:.1f} ms)'
)
print()
if node_floor.min > prettier_rows[0][1]['Python'].min:
    print('Starting node with an empty script takes longer than the Python')
    print('implementation takes to rewrite one file and exit.')
    print()

if unmatched:
    print('The three programs did not return the same answer. Ignore this table:')
    for problem in unmatched:
        print(f'     {problem}')
else:
    print('All three returned the same bytes for each document read on stdin, on')
    print('every repetition, and each of them named every file in the rows that')
    print('pass files by name.')

unsteady = [
    f'{name}, {label}'
    for label, timings in prettier_rows
    for name, timing in timings.items()
    if len(timing.digests) > 1 or len(timing.codes) > 1
]
if unsteady:
    print('\nA command did not return the same result on every repetition:')
    for entry in unsteady:
        print(f'     {entry}')

noisy = [
    (name, label, timing)
    for label, timings in prettier_rows
    for name, timing in timings.items()
    if timing.spread > SPREAD_LIMIT and not timing.spawn_bound
]
if noisy:
    print(f'\nIn these rows the median is more than {SPREAD_LIMIT:.0f}% above the')
    print('minimum, and the row measures more than process startup, so the')
    print('machine was busy. Measure them again:')
    for name, label, timing in noisy:
        print(f'     {name:<9} {label:<44} +{timing.spread:.0f}%')
else:
    print('\nEvery row that measures more than starting a process is within')
    print(f'{SPREAD_LIMIT:.0f}% of its own minimum.')

COLORS = {'Rust': '#ff7f0e', 'Python': '#1f77b4', 'Prettier': '#9467bd'}
BAR = 0.26

figure, axes = plt.subplots(figsize=(9, 4.6))
for offset, (name, _) in enumerate(PROGRAMS):
    positions = [index + (offset - 1) * BAR for index in range(len(prettier_rows))]
    values = [timings[name].min for _, timings in prettier_rows]
    axes.barh(positions, values, height=BAR, color=COLORS[name], label=name)
    for position, value in zip(positions, values, strict=True):
        axes.text(value * 1.15, position, f'{value:,.1f} ms', va='center', fontsize=8)
axes.axvline(SPAWN_FLOOR, color='#444444', linestyle='--', linewidth=1)
axes.set_yticks(range(len(prettier_rows)), [label for label, _ in prettier_rows])
axes.set_xscale('log')
axes.set_xlim(
    right=max(timing.min for _, timings in prettier_rows for timing in timings.values())
    * 8
)
axes.set_xlabel('time (milliseconds, logarithmic scale)')
axes.set_title('The dashed line is what starting any process costs')
axes.invert_yaxis()
axes.grid(alpha=0.3, axis='x')
axes.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.16),
    ncol=3,
    frameon=False,
    fontsize=9,
)
hide_spines(axes)
figure.tight_layout()
print()
save_chart(figure, 'benchmarks-prettier.svg')

| what is being processed | Rust | Python | Prettier | Prettier / Rust | Prettier / Python |
| -- | --: | --: | --: | --: | --: |
| one file | 1.9 / 2.4 ms | 30.4 / 33.3 ms | 86.3 / 95.6 ms | ≥46.4x | 2.8x |
| a typical commit: 5 files | 1.7 / 1.9 ms | 28.6 / 30.9 ms | 101.7 / 108.9 ms | ≥59.1x | 3.6x |
| a large repository: 2000 files | 38.4 / 40.8 ms | 180.0 / 197.2 ms | 1955.1 / 2133.3 ms | 51.0x | 10.9x |
| one 4 MB document | 107.2 / 107.9 ms | 888.9 / 948.5 ms | 4198.5 / 4288.6 ms | 39.2x | 4.7x |

Each time is the minimum and then the median. ≥ means the Rust time is within 2 times the 1.13 ms floor, so that ratio is a lower bound.

prettier  3.9.9
node      v26.3.0
          42.0 ms to run an empty script (median 43.5 ms)

Starting node with an empty script takes longer than the Python
implementation takes to rewrite one file and exit.

All three returned the same bytes for each document read on stdin, on
every repetition, and each of them named every file in the rows that
pass files by name.

Every row that measures more than starting a process is within
15% of its own minimum.

chart written to docs/notebooks/benchmarks/benchmarks-prettier.svg


![Rust, Python and Prettier on the same documents](benchmarks-prettier.svg)

## When the answers differ

Every document above is one the three programs return as the same bytes, so
those rows compare the same work. Where Prettier returns a different document,
it has done different work, and a time for it answers a different question.
Those documents are timed here, separately, and never added to the rows above.

The cell below builds two sets of files that differ in one respect. Every file
opens with three label rows, each a bold key and its value on a line of its
own, and then holds the same wrapped paragraphs as the files above. In the
first set, a blank line separates the label rows, and all three programs return
the same bytes. In the second set, the label rows are on adjacent lines. This
tool keeps each label row on its own line, while Prettier joins adjacent lines
into one paragraph, so it returns a different document. The cell checks both of
those outcomes before it times anything.

The two sets differ by two bytes a file, so a difference between their times is
the cost of the disagreement itself rather than of more text.

In [10]:
# Three label rows on adjacent lines, and the same rows with a blank line
# between each pair. Only the second shape is one Prettier joins.
ADJACENT = '**Date:** 2026-01-01\n**Status:** draft\n**Owner:** docs\n\n'
SPACED = ADJACENT.replace('\n**', '\n\n**')

label_sets = (
    ('the answers match', SPACED, True),
    ('Prettier returns a different document', ADJACENT, False),
)

label_rows: list[tuple[str, dict[str, Timing]]] = []
label_problems: list[str] = []
for label, head, prettier_agrees in label_sets:
    folder = scratch / ('labels-spaced' if prettier_agrees else 'labels-adjacent')
    rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True)
    text = wrapped_document(3, head)
    paths = [str(folder / f'{index}.md') for index in range(2000)]
    for path in paths:
        Path(path).write_text(text)

    answers = {
        name: subprocess_run(
            on_stdin(name, command, Path(paths[0])),
            input=text.encode(),
            capture_output=True,
            check=False,
        ).stdout
        for name, command in PROGRAMS
    }
    if answers['Rust'] != answers['Python']:
        label_problems.append(f'the two implementations differ: {label}')
    if (answers['Prettier'] == answers['Rust']) != prettier_agrees:
        label_problems.append(f'prettier did not answer as expected: {label}')

    timings = {}
    for name, command in PROGRAMS:
        if not names_every_file(name, command, paths):
            label_problems.append(f'{name} did not name every file: {label}')
        timings[name] = measure(by_name(name, command, paths), LARGE_REPS)
    label_rows.append((label, timings))

table = [
    (
        '| 2000 files, where | Rust | Python | Prettier '
        '| Prettier / Rust | Prettier / Python |'
    ),
    '| -- | --: | --: | --: | --: | --: |',
]
append_to_table = table.append
for label, timings in label_rows:
    cells = ' | '.join(
        f'{timings[name].min:.1f} / {timings[name].median:.1f} ms'
        for name, _ in PROGRAMS
    )
    append_to_table(
        f'| {label} | {cells} '
        f'| {multiple(timings["Prettier"], timings["Rust"])} '
        f'| {multiple(timings["Prettier"], timings["Python"])} |'
    )
append_to_table('')
append_to_table('Each time is the minimum and then the median.')
display(Markdown(chr(10).join(table)))

if label_problems:
    print('The two sets did not behave as built. Ignore this table:')
    for problem in label_problems:
        print(f'     {problem}')
else:
    print('In the first set all three returned the same bytes. In the second, the')
    print('two implementations returned the same bytes and Prettier a different')
    print('document. Every program named every file in both sets.')

(_, matching), (_, differing) = label_rows
print('\nWhat a file costs where the answers differ, against where they match:')
for name, _ in PROGRAMS:
    extra = (differing[name].min - matching[name].min) / 2000 * 1000
    print(f'     {name:<9} {extra:+.1f} us per file')
before = matching['Prettier'].min / matching['Rust'].min
after = differing['Prettier'].min / differing['Rust'].min
print(
    f'\nPrettier takes {after:.0f} times as long as Rust where the answers '
    f'differ,\nand {before:.0f} times as long where they match.'
)

| 2000 files, where | Rust | Python | Prettier | Prettier / Rust | Prettier / Python |
| -- | --: | --: | --: | --: | --: |
| the answers match | 41.5 / 44.7 ms | 210.3 / 220.8 ms | 2215.8 / 2403.4 ms | 53.3x | 10.5x |
| Prettier returns a different document | 46.5 / 47.6 ms | 207.2 / 235.6 ms | 2114.6 / 2315.4 ms | 45.5x | 10.2x |

Each time is the minimum and then the median.

In the first set all three returned the same bytes. In the second, the
two implementations returned the same bytes and Prettier a different
document. Every program named every file in both sets.

What a file costs where the answers differ, against where they match:
     Rust      +2.5 us per file
     Python    -1.5 us per file
     Prettier  -50.6 us per file

Prettier takes 45 times as long as Rust where the answers differ,
and 53 times as long where they match.


## Through pre-commit

Run as a `pre-commit` hook, each program also pays for the work `pre-commit`
does around every hook: it starts, reads its configuration, checks that the
hook's environment is still installed, and splits the files among processes.
The cell below measures that cost along with the hooks.

The hooks are defined in a scratch repository. The Python hook takes its entry
and its language from this repository's Python mirror and installs this
checkout. The Prettier hook restates the one the Prettier mirror publishes, a
`language: node` hook whose one dependency is Prettier at the pinned version.
The Rust hook is a substitution. The Rust mirror builds the crate published on
crates.io rather than this checkout, so the release binary runs as a
`language: system` hook instead. The cell checks that the `pre-commit` it runs
gives a Rust environment its basic check, which starts no process, so the
substitution changes how the program was built and not how long a run takes. A
fourth hook runs `true`, which measures `pre-commit` alone.

Each hook is first given a wrapped file, and the cell checks that all three
rewrite it to the same bytes. The files it times are already joined, so every
hook leaves them alone. These hooks rewrite files, and a file rewritten on the
first repetition would be a different file on every repetition after it.

In [11]:
import shlex

import yaml
from pre_commit import lang_base
from pre_commit.languages import rust as pre_commit_rust

PRE_COMMIT = REPO / '.venv/bin/pre-commit'
PRE_COMMIT_VERSION = subprocess_run(
    [str(PRE_COMMIT), '--version'], capture_output=True, text=True, check=False
).stdout.strip()

# A system hook stands in for the Rust one only while pre-commit gives a Rust
# environment its basic check, which starts no process.
if pre_commit_rust.health_check is not lang_base.basic_health_check:
    raise SystemExit(f'{PRE_COMMIT_VERSION} checks a Rust environment another way')


def manifest(mirror: str, hook_id: str) -> dict[str, object]:
    """Return one hook as the named mirror's manifest declares it."""
    hooks = yaml.safe_load(
        (REPO / 'mirrors' / mirror / 'pre-commit-hooks.yaml').read_text(
            encoding='utf-8'
        )
    )
    return next(hook for hook in hooks if hook['id'] == hook_id)


python_hook = manifest('py', 'unwrap-markdown-prose-py')
rust_hook = manifest('rs', 'unwrap-markdown-prose-rs')
_, *rust_args = shlex.split(str(rust_hook['entry']))
config = {
    'repos': [
        {
            'repo': 'local',
            'hooks': [
                {
                    'id': 'nothing',
                    'name': 'nothing',
                    'entry': 'true',
                    'language': 'system',
                    'types': ['markdown'],
                },
                {
                    'id': 'Rust',
                    'name': 'Rust',
                    'entry': shlex.join([str(RUST_CLI), *rust_args]),
                    'language': 'system',
                    'types': rust_hook['types'],
                },
                {
                    'id': 'Python',
                    'name': 'Python',
                    'entry': python_hook['entry'],
                    'language': python_hook['language'],
                    'types': python_hook['types'],
                    'additional_dependencies': [str(REPO)],
                },
                # What rbubley/mirrors-prettier publishes for its one hook, at
                # the pinned version, with the setting this comparison is about
                # passed as an argument.
                {
                    'id': 'Prettier',
                    'name': 'Prettier',
                    'entry': 'prettier --write --ignore-unknown',
                    'language': 'node',
                    'types': ['text'],
                    'additional_dependencies': [f'prettier@{PRETTIER_PIN}'],
                    'args': ['--prose-wrap', 'never'],
                },
            ],
        }
    ]
}

hook_repo = scratch / 'hook-repo'
# Its own store, so nothing here reads or fills the one in the home directory.
hook_home = scratch / 'pre-commit-home'
for folder in (hook_repo, hook_home):
    rmtree(folder, ignore_errors=True)
hook_repo.mkdir(parents=True)
hook_env = {**os.environ, 'PRE_COMMIT_HOME': str(hook_home)}

wrapped = wrapped_document(3)
joined = subprocess_run(
    [str(RUST_CLI), '-'], input=wrapped.encode(), capture_output=True, check=False
).stdout
hook_files = [f'{index}.md' for index in range(5)]
for name in hook_files:
    (hook_repo / name).write_bytes(joined)
(hook_repo / '.pre-commit-config.yaml').write_text(
    yaml.safe_dump(config, sort_keys=False)
)
for argv in (
    ['git', 'init', '--quiet'],
    ['git', 'add', '--all'],
    [str(PRE_COMMIT), 'install-hooks'],
):
    done = subprocess_run(
        argv, cwd=hook_repo, env=hook_env, capture_output=True, text=True, check=False
    )
    if done.returncode != 0:
        raise SystemExit(f'{shlex.join(argv)} failed: {done.stderr}{done.stdout}')

# A hook that passed because it never touched the file would be timed as a
# fast hook, so each one first has to rewrite a wrapped file to the same bytes.
canary = hook_repo / 'canary.md'
canary_problems = []
for name, _ in PROGRAMS:
    canary.write_text(wrapped)
    subprocess_run(
        [str(PRE_COMMIT), 'run', name, '--files', canary.name],
        cwd=hook_repo,
        env=hook_env,
        capture_output=True,
        check=False,
    )
    if canary.read_bytes() != joined:
        canary_problems.append(name)
canary.unlink()

HOOK_IDS = ('nothing', *(name for name, _ in PROGRAMS))
FILE_COUNTS = 1, len(hook_files)
hook_timings = {
    (hook, count): measure(
        [str(PRE_COMMIT), 'run', hook, '--files', *hook_files[:count]],
        cwd=hook_repo,
        env=hook_env,
    )
    for count in FILE_COUNTS
    for hook in HOOK_IDS
}

table = [
    '| files | pre-commit alone | Rust hook | Python hook | Prettier hook |',
    '| --: | --: | --: | --: | --: |',
]
append_to_table = table.append
for count in FILE_COUNTS:
    alone = hook_timings['nothing', count].min
    cells = ' | '.join(
        f'{hook_timings[name, count].min:.0f} ms '
        f'({hook_timings[name, count].min - alone:+.0f})'
        for name, _ in PROGRAMS
    )
    append_to_table(f'| {count} | {alone:.0f} ms | {cells} |')
append_to_table('')
append_to_table(
    'Each time is the minimum. The figure in parentheses is what the hook adds to '
    '`pre-commit` alone, which subtracting measures reliably for the reason the '
    'startup section gives: the cost of `pre-commit` itself is in both numbers.'
)
display(Markdown(chr(10).join(table)))

print(PRE_COMMIT_VERSION)
print()
one_file = {name: hook_timings[name, 1].min for name in HOOK_IDS}
for name in ('Rust', 'Python'):
    gap = one_file['Prettier'] - one_file[name]
    side = 'longer' if gap > 0 else 'less'
    print(
        f'On one file, the Prettier hook takes {abs(gap):.0f} ms {side} '
        f'than the {name} hook.'
    )

untouched = all((hook_repo / name).read_bytes() == joined for name in hook_files)
clean = all(timing.codes == frozenset({0}) for timing in hook_timings.values())
steady = all(len(timing.digests) == 1 for timing in hook_timings.values())
print()
if canary_problems:
    print('These hooks did not rewrite a wrapped file to the same bytes as the')
    print('others, so their times are not comparable:')
    for name in canary_problems:
        print(f'     {name}')
else:
    print('Each hook rewrote a wrapped file to the same bytes as the others.')
if untouched and clean and steady:
    print('Every timed run passed, printed the same thing, and left the files as')
    print('they were.')
else:
    print('A timed run failed, printed something different, or changed a file,')
    print('so these rows are worth measuring again.')

figure, axes = plt.subplots(figsize=(8, 3.0))
hook_labels = ['pre-commit alone', *(f'{name} hook' for name, _ in PROGRAMS)]
alone = one_file['nothing']
totals = [alone, *(one_file[name] for name, _ in PROGRAMS)]
axes.barh(hook_labels, [alone] * len(hook_labels), color='#bbbbbb')
axes.barh(
    hook_labels[1:],
    [max(total - alone, 0) for total in totals[1:]],
    left=alone,
    color=[COLORS[name] for name, _ in PROGRAMS],
)
for index, total in enumerate(totals):
    axes.text(total + max(totals) * 0.02, index, f'{total:.0f} ms', va='center')
axes.set_xlim(right=max(totals) * 1.2)
axes.set_xlabel('time to run one hook on one file (milliseconds)')
axes.set_title('What each hook adds to pre-commit, in color')
axes.invert_yaxis()
axes.grid(alpha=0.3, axis='x')
hide_spines(axes)
figure.tight_layout()
print()
save_chart(figure, 'benchmarks-pre-commit.svg')

| files | pre-commit alone | Rust hook | Python hook | Prettier hook |
| --: | --: | --: | --: | --: |
| 1 | 98 ms | 104 ms (+6) | 140 ms (+42) | 209 ms (+110) |
| 5 | 106 ms | 98 ms (-8) | 151 ms (+45) | 228 ms (+123) |

Each time is the minimum. The figure in parentheses is what the hook adds to `pre-commit` alone, which subtracting measures reliably for the reason the startup section gives: the cost of `pre-commit` itself is in both numbers.

pre-commit 4.6.2

On one file, the Prettier hook takes 105 ms longer than the Rust hook.
On one file, the Prettier hook takes 69 ms longer than the Python hook.

Each hook rewrote a wrapped file to the same bytes as the others.
Every timed run passed, printed the same thing, and left the files as
they were.

chart written to docs/notebooks/benchmarks/benchmarks-pre-commit.svg


![What each hook adds to pre-commit on one file](benchmarks-pre-commit.svg)

## What the measurements add up to

The last cell collects the figures above into the statements they support, and
adds one derived number: the file count at which the per-file cost overtakes
the cost of starting the program.

It recommends nothing. Which implementation to use is a short rule that turns
on what is already installed rather than on any measurement here, and it lives
in the README.

In [12]:
import textwrap

# The file count at which the two halves of the total are equal: fixed cost in
# milliseconds against per-file cost in microseconds.
crossover = {
    name: count_intercept[name] * 1000 / count_marginal[name]
    for name in ('Python', 'Rust')
}

# Prettier's multiple over each implementation, row by row in its table.
over_python = [
    timings['Prettier'].min / timings['Python'].min for _, timings in prettier_rows
]
over_rust = [
    timings['Prettier'].min / timings['Rust'].min for _, timings in prettier_rows
]
widest = max(range(len(over_rust)), key=over_rust.__getitem__)
rust_bound = (
    ', and the larger figure is a lower bound'
    if prettier_rows[widest][1]['Rust'].spawn_bound
    else ''
)
label_multiple = {
    label: timings['Prettier'].min / timings['Rust'].min
    for label, timings in label_rows
}
alone_one = hook_timings['nothing', 1].min

large_row = results[-1]
large_ratio = large_row.python.min / large_row.rust.min
startup_ratio = python_startup / rust_startup
bound = BOUND if rust_startup < SPAWN_FLOOR * FLOOR_FACTOR else ''

document = [
    '### What was measured',
    '',
    f'- **Starting the program.** Python takes {bound}{startup_ratio:.0f} times as '
    f'long, a difference of {saving:.0f} ms on every run.',
    f'- **Each additional file.** Python takes between {min(size_ratios):.1f} and '
    f'{max(size_ratios):.1f} times as long, depending on how much text the file '
    f'holds.',
    f'- **One {large.stat().st_size / 1e6:.0f} MB document**, where starting the '
    f'program is a negligible part of the total: Python takes '
    f'{large_ratio:.1f} times as long.',
    f'- **Setting up an environment.** Python {python_install.min:.2f} s, Rust '
    f'{rust_environment_cost.min:.2f} s'
    + (
        ', which is the same to within the accuracy of the measurement.'
        if abs(extra_install) / cheaper < SAME_WITHIN
        else '.'
    ),
    f'- **Where the balance tips.** Below about {crossover["Python"]:.0f} files in '
    f'one run, most of the Python total is starting the program; above it, most is '
    f'the per-file cost. For Rust, that point is about '
    f'{crossover["Rust"]:.0f} files.',
    f'- **Prettier, on the same documents.** Prettier takes between '
    f'{min(over_python):.1f} and {max(over_python):.1f} times as long as the Python '
    f'implementation, and between {min(over_rust):.0f} and {max(over_rust):.0f} '
    f'times as long as the Rust one{rust_bound}.',
    f'- **When the answers differ.** Where Prettier returns a different document, '
    f'it takes {label_multiple[label_sets[1][0]]:.0f} times as long as Rust, '
    f'against {label_multiple[label_sets[0][0]]:.0f} times on files of the same '
    f'size where the answers match.',
    f'- **Through `pre-commit`.** `pre-commit` alone takes {alone_one:.0f} ms '
    f'on one file. The Rust hook adds '
    f'{hook_timings["Rust", 1].min - alone_one:.0f} ms to that, the Python hook '
    f'{hook_timings["Python", 1].min - alone_one:.0f} ms, and the Prettier hook '
    f'{hook_timings["Prettier", 1].min - alone_one:.0f} ms.',
    '',
    '### What was not measured',
    '',
    textwrap.fill(
        'Installing a Rust toolchain, which is what `pre-commit` does before it can '
        'build a `language: rust` hook on a machine without cargo. It costs far more '
        'than either number above, and it is the reason the rule in the README turns '
        'on whether cargo is already installed rather than on any figure here.',
        width=86,
    ),
    '',
    textwrap.fill(
        'Installing Prettier. A `language: node` hook installs it with npm into an '
        'environment `pre-commit` builds, and on a machine without Node it '
        'downloads Node first, so that cost turns on what is already installed in '
        'the same way the Rust one does.',
        width=86,
    ),
]

if unmatched or label_problems or canary_problems:
    document += [
        '',
        '**The Prettier sections failed a check above. Ignore those figures.**',
    ]

if not AGREE:
    document += [
        '',
        '**The implementations returned different results above. Ignore all of this.**',
    ]

display(Markdown(chr(10).join(document)))

### What was measured

- **Starting the program.** Python takes ≥16 times as long, a difference of 26 ms on every run.
- **Each additional file.** Python takes between 3.4 and 9.6 times as long, depending on how much text the file holds.
- **One 4 MB document**, where starting the program is a negligible part of the total: Python takes 10.3 times as long.
- **Setting up an environment.** Python 2.08 s, Rust 2.47 s, which is the same to within the accuracy of the measurement.
- **Where the balance tips.** Below about 371 files in one run, most of the Python total is starting the program; above it, most is the per-file cost. For Rust, that point is about 85 files.
- **Prettier, on the same documents.** Prettier takes between 2.8 and 10.9 times as long as the Python implementation, and between 39 and 59 times as long as the Rust one, and the larger figure is a lower bound.
- **When the answers differ.** Where Prettier returns a different document, it takes 45 times as long as Rust, against 53 times on files of the same size where the answers match.
- **Through `pre-commit`.** `pre-commit` alone takes 98 ms on one file. The Rust hook adds 6 ms to that, the Python hook 42 ms, and the Prettier hook 110 ms.

### What was not measured

Installing a Rust toolchain, which is what `pre-commit` does before it can build a
`language: rust` hook on a machine without cargo. It costs far more than either number
above, and it is the reason the rule in the README turns on whether cargo is already
installed rather than on any figure here.

Installing Prettier. A `language: node` hook installs it with npm into an environment
`pre-commit` builds, and on a machine without Node it downloads Node first, so that
cost turns on what is already installed in the same way the Rust one does.